# 01 – Meldedaten einlesen und zusammenführen

**Ziel:** Aus den 12 SurvStat-Exporten (Influenza 2015 bis 2026) wird **eine** Tabelle mit einer Zeile pro **Erreger × Kreis × Meldewoche** und der Zahl der gemeldeten Fälle.

**Ergebnis:** `02_Daten/aufbereitet/meldedaten_kreis_woche.csv`. Mit dieser Datei arbeiten alle weiteren Notebooks.

### So startest du das Notebook
1. Terminal öffnen und in den Projektordner wechseln:
   `cd ~/Desktop/Claude/"Epi Karnevalseffekt"`
2. Jupyter starten: `.venv/bin/jupyter lab`
3. Im Browser links `04_Skripte/01_Meldedaten_einlesen.ipynb` öffnen.
4. Zellen nacheinander mit **Shift + Enter** ausführen. Jede Zelle hat darüber eine Erklärung.

`.venv` ist eine eigene Python-Umgebung nur für dieses Projekt (mit pandas, matplotlib und Jupyter). So stört nichts den Rest deines Macs.

## Schritt 1: Werkzeuge laden und Ordner festlegen

- **pandas** ist die Standardbibliothek für Tabellen in Python, vergleichbar mit Excel, aber per Code und damit nachvollziehbar.
- **matplotlib** zeichnet Diagramme.
- Die Pfade sind *relativ* zum Ordner `04_Skripte`, in dem dieses Notebook liegt.

In [ ]:
from pathlib import Path
import datetime as dt

import pandas as pd
import matplotlib.pyplot as plt

ORDNER_ROH = Path("../02_Daten/SurvStat")      # die 12 Exporte
ORDNER_AUS = Path("../02_Daten/aufbereitet")   # hier landet das Ergebnis
ORDNER_AUS.mkdir(exist_ok=True)

dateien = sorted(ORDNER_ROH.glob("*.csv"))
print(len(dateien), "Dateien gefunden")

## Schritt 2: Eine Datei ansehen

Die SurvStat-Dateien haben drei Besonderheiten, die wir beim Einlesen angeben müssen:
- **Zeichensatz UTF-16** (`encoding="utf-16"`), sonst werden Umlaute falsch gelesen
- **Tabulator** als Trennzeichen (`sep="\t"`)
- **Zwei Kopfzeilen:** Zeile 1 enthält nur „Kreis / Meldewoche“, Zeile 2 die Wochennummern. Wir überspringen Zeile 1 (`skiprows=1`).

Leere Felder bedeuten **0 Fälle**. pandas liest sie zunächst als `NaN` (fehlender Wert).

In [ ]:
beispiel = pd.read_csv(ORDNER_ROH / "Influenza_2024.csv", encoding="utf-16", sep="\t", skiprows=1)
beispiel.iloc[:5, :8]

## Schritt 3: Vom „breiten“ ins „lange“ Format

SurvStat liefert eine **breite** Tabelle: eine Zeile pro Kreis, eine Spalte pro Woche.
Für das Modell brauchen wir das **lange** Format: **eine Zeile pro Kreis und Woche**, wie in der Vorlesung („eine Zeile = Region × Zeitpunkt“).

Das erledigt `melt()`. Die Funktion `lese_survstat()` macht das für eine Datei:
1. Datei einlesen
2. erste Spalte in „Kreis“ umbenennen
3. ins lange Format umbauen
4. leere Felder → 0
5. Erreger und Jahr aus dem Dateinamen ergänzen (`Influenza_2024` → Influenza, 2024)

In [ ]:
def lese_survstat(datei):
    erreger, jahr = datei.stem.split("_")
    tabelle = pd.read_csv(datei, encoding="utf-16", sep="\t", skiprows=1)
    tabelle = tabelle.rename(columns={tabelle.columns[0]: "Kreis"})
    lang = tabelle.melt(id_vars="Kreis", var_name="Meldewoche", value_name="Faelle")
    lang["Faelle"] = lang["Faelle"].fillna(0).astype(int)
    lang["Meldewoche"] = lang["Meldewoche"].astype(int)
    lang["Meldejahr"] = int(jahr)
    lang["Erreger"] = erreger
    return lang

lese_survstat(ORDNER_ROH / "Influenza_2024.csv").head()

## Schritt 4: Alle 12 Dateien einlesen und untereinanderhängen

`pd.concat()` hängt die Tabellen der einzelnen Dateien untereinander.
Zur Kontrolle: die Summe der Fälle je Datei. Sie muss mit SurvStat übereinstimmen (z. B. Influenza 2024: 218.307).

In [ ]:
daten = pd.concat([lese_survstat(d) for d in dateien], ignore_index=True)

print(daten.shape[0], "Zeilen")
daten.groupby(["Erreger", "Meldejahr"])["Faelle"].sum().unstack(0)

## Schritt 5: Berlin zusammenfassen und „Unbekannt“ entfernen

- SurvStat teilt **Berlin in 12 Bezirke** auf. Bevölkerungs- und Wetterdaten gibt es aber für Berlin als Ganzes. Deshalb fassen wir die Bezirke zu **„SK Berlin“** zusammen (Fälle addieren).
- Die Zeile **„Unbekannt“** enthält Fälle ohne bekannten Kreis. Sie lassen sich keinem Ort zuordnen und fallen weg.

Danach sollten es **400 Kreise** sein. Das ist die Zahl der Kreise und kreisfreien Städte in Deutschland.

In [ ]:
daten.loc[daten["Kreis"].str.startswith("SK Berlin"), "Kreis"] = "SK Berlin"
daten = daten[daten["Kreis"] != "Unbekannt"]
daten = daten.groupby(["Erreger", "Kreis", "Meldejahr", "Meldewoche"], as_index=False)["Faelle"].sum()

print("Anzahl Kreise:", daten["Kreis"].nunique())

## Schritt 6: Jeder Woche ein Datum geben

Die Meldewoche zählt nach Kalenderwochen (ISO 8601): Woche 1 ist die Woche mit dem ersten Donnerstag des Jahres. Darum haben 2015, 2020 und 2026 eine Woche 53, und SurvStat liefert genau für diese Jahre 53 Spalten.

Wir geben jeder Woche das Datum ihres **Montags**. Damit lassen sich die Wochen später einfach mit den Karnevalsterminen vergleichen.

**Wichtig für das laufende Jahr:** Die Daten reichen nur bis zur letzten abgeschlossenen Woche vor dem Abruf. Spätere Wochen stehen in der Datei als leer, also 0, obwohl sie noch gar nicht stattgefunden haben. Diese Wochen entfernen wir. Als Abrufdatum gilt das Änderungsdatum der neuesten Datei. So liefert das Notebook auch an einem späteren Tag ausgeführt denselben Stand.

In [ ]:
daten["Wochenbeginn"] = pd.to_datetime([
    dt.date.fromisocalendar(jahr, woche, 1)
    for jahr, woche in zip(daten["Meldejahr"], daten["Meldewoche"])
])

# Letzte vollständige Meldewoche: die Woche vor der Woche des Abrufs (Abruf = Änderungsdatum der neuesten Datei)
abruf = dt.date.fromtimestamp(max(d.stat().st_mtime for d in dateien))
LETZTE_WOCHE = pd.Timestamp(abruf - dt.timedelta(days=abruf.weekday() + 7))
print("Abruf am", abruf, "| letzte vollständige Meldewoche ab", LETZTE_WOCHE.date())
daten = daten[daten["Wochenbeginn"] <= LETZTE_WOCHE]

daten = daten.sort_values(["Erreger", "Kreis", "Wochenbeginn"]).reset_index(drop=True)
daten.head()

## Schritt 7: Unsere Städte finden

Die Kreise heißen in SurvStat nicht immer so, wie man sie sucht. Drei Besonderheiten:
- **Aachen** gibt es nur als **„StädteRegion Aachen“** (Stadt und Umland zusammen).
- **Hannover** gibt es nur als **„Region Hannover“**.
- **Neuss** gehört zum **„LK Rhein-Kreis Neuss“**.

Das ist wichtig für die Auswahl der Hochburgen und Kontrollkreise und muss in der Methodik stehen.

In [ ]:
staedte = ["Köln", "Düsseldorf", "Bonn", "Aachen", "Mainz", "Mönchengladbach", "Krefeld", "Neuss",
           "Hamburg", "Berlin", "Bremen", "Hannover", "Leipzig", "Dresden", "Nürnberg", "Münster", "Bielefeld"]
alle_kreise = pd.Series(sorted(daten["Kreis"].unique()))
alle_kreise[alle_kreise.str.contains("|".join(staedte))].tolist()

## Schritt 8: Ein erster Blick

Influenza-Fälle pro Woche in Köln und Hamburg. Achte auf:
- die **Grippewellen** jeden Winter
- die fast fehlende Welle **2020/21** (Corona-Maßnahmen)
- die sehr starke Welle **2024/25**

Das sind noch **absolute Fallzahlen**. Köln und Hamburg sind unterschiedlich groß. Vergleichbar werden sie erst als Inzidenz (Fälle je 100.000 Einwohner). Das kommt im nächsten Schritt mit den Bevölkerungsdaten.

In [ ]:
auswahl = daten[(daten["Erreger"] == "Influenza") & daten["Kreis"].isin(["SK Köln", "SK Hamburg"])]

fig, ax = plt.subplots(figsize=(12, 4))
for kreis, teil in auswahl.groupby("Kreis"):
    ax.plot(teil["Wochenbeginn"], teil["Faelle"], label=kreis, linewidth=1)
ax.set_ylabel("gemeldete Fälle pro Woche")
ax.set_title("Influenza – gemeldete Fälle pro Meldewoche")
ax.legend()
plt.show()

## Schritt 9: Ergebnis speichern

Die fertige Tabelle hat diese Spalten:

| Spalte | Bedeutung |
|---|---|
| Erreger | Influenza |
| Kreis | Name des Kreises laut SurvStat |
| Meldejahr, Meldewoche | Kalenderwoche nach ISO 8601 |
| Faelle | gemeldete Fälle in dieser Woche |
| Wochenbeginn | Datum des Montags dieser Woche |

In [ ]:
datei_aus = ORDNER_AUS / "meldedaten_kreis_woche.csv"
daten.to_csv(datei_aus, index=False)
print("Gespeichert:", datei_aus, "|", len(daten), "Zeilen")

## Was du für die Methodik mitnehmen kannst

- Datenquelle: RKI SurvStat@RKI 2.0, Abfragedatum 13.11.2026 (Jahre 2025 und 2026 am 15.11.2026 neu abgerufen), Referenzdefinition „Ja“, Meldeweg über Gesundheitsamt und Landesstelle
- Influenza 2015 bis KW 45/2026, je Kreis und Meldewoche (COVID-19 nicht mehr enthalten, die Exporte liegen in `99_Archiv/02_Daten/SurvStat_COVID`)
- Berliner Bezirke zu Berlin zusammengefasst, Fälle mit unbekanntem Kreis ausgeschlossen
- Ergebnis: 400 Kreise, eine Zeile pro Erreger, Kreis und Woche